In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
PASTA = pathlib.Path('/content/drive/MyDrive/precos-combustiveis-anp/dados_brutos')

# conferência: devem aparecer os arquivos baixados
print(sorted(p.name for p in PASTA.iterdir()))

Mounted at /content/drive
[]


In [4]:
import zipfile
arq = PASTA / "ca-2025-01.zip"
with zipfile.ZipFile(arq) as z:
    print(z.namelist())
    nome = z.namelist()[0]
    print(z.open(nome).read(700))   # primeiros bytes crus

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/precos-combustiveis-anp/dados_brutos/ca-2025-01.zip'

In [5]:
print(sorted(p.name for p in PASTA.iterdir()))

[]


In [6]:
print(sorted(p.name for p in PASTA.iterdir()))

[]


In [7]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

import pathlib
PASTA = pathlib.Path('/content/drive/MyDrive/precos-combustiveis-anp/dados_brutos')
print(sorted(p.name for p in PASTA.iterdir()))

Mounted at /content/drive
[]


In [8]:
import os
print(os.listdir('/content/drive/MyDrive'))
print(os.listdir('/content/drive/MyDrive/precos-combustiveis-anp'))

['Colab Notebooks', 'relogio_da_justica.csv', 'relogio_da_justica_dados.gsheet', 'radar_emprego_formal', 'precos-combustiveis-anp', 'precos-combustiveis-anp dados_brutos']
['dados_brutos']


In [9]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

import pathlib
PASTA = pathlib.Path('/content/drive/MyDrive/precos-combustiveis-anp/dados_brutos')
print(sorted(p.name for p in PASTA.iterdir()))

Mounted at /content/drive
['01_baixar_dados.ipynb', '02_inspecao.ipynb', 'ca-2023-01.zip', 'ca-2023-02.zip', 'ca-2024-01.zip', 'ca-2024-02.zip', 'ca-2025-01.zip', 'ca-2025-02.zip', 'ca-2026-01.zip']


In [10]:
import zipfile, io, re, unicodedata
import pandas as pd

def limpa_nome(c):
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_")

def abrir_bytes(caminho):
    """Devolve (nome do csv, conteúdo em bytes), seja .zip ou .csv."""
    if caminho.suffix == ".zip":
        with zipfile.ZipFile(caminho) as z:
            nomes = [n for n in z.namelist() if n.lower().endswith(".csv")]
            return nomes[0], z.read(nomes[0])
    return caminho.name, caminho.read_bytes()

def detectar(data):
    """Descobre codificação, separador e cabeçalho."""
    if data.startswith(b"\xef\xbb\xbf"):
        enc = "utf-8-sig"
    else:
        try:
            data.decode("utf-8"); enc = "utf-8"
        except UnicodeDecodeError:
            enc = "latin-1"
    cab = data.split(b"\n", 1)[0].decode(enc, errors="replace").strip()
    sep = max([";", ",", "\t", "|"], key=cab.count)
    return enc, sep, cab

In [11]:
arquivos = sorted(p for p in PASTA.iterdir() if p.name.startswith("ca-"))
for arq in arquivos:
    nome, data = abrir_bytes(arq)
    enc, sep, cab = detectar(data)
    print(arq.name, "|", nome, "|", round(len(data)/1e6), "MB |", enc, "| sep:", repr(sep))
    print("   ", cab[:300])

ca-2023-01.zip | Preços semestrais - AUTOMOTIVOS_2023.01.csv | 74 MB | utf-8-sig | sep: ';'
    Regiao - Sigla;Estado - Sigla;Municipio;Revenda;CNPJ da Revenda;Nome da Rua;Numero Rua;Complemento;Bairro;Cep;Produto;Data da Coleta;Valor de Venda;Valor de Compra;Unidade de Medida;Bandeira
ca-2023-02.zip | Preços semestrais - AUTOMOTIVOS_2023.02.csv | 81 MB | utf-8-sig | sep: ';'
    Regiao - Sigla;Estado - Sigla;Municipio;Revenda;CNPJ da Revenda;Nome da Rua;Numero Rua;Complemento;Bairro;Cep;Produto;Data da Coleta;Valor de Venda;Valor de Compra;Unidade de Medida;Bandeira
ca-2024-01.zip | Preços semestrais - AUTOMOTIVOS_2024.01.csv | 81 MB | utf-8-sig | sep: ';'
    Regiao - Sigla;Estado - Sigla;Municipio;Revenda;CNPJ da Revenda;Nome da Rua;Numero Rua;Complemento;Bairro;Cep;Produto;Data da Coleta;Valor de Venda;Valor de Compra;Unidade de Medida;Bandeira
ca-2024-02.zip | Preços semestrais - AUTOMOTIVOS_2024.02.csv | 72 MB | utf-8-sig | sep: ';'
    Regiao - Sigla;Estado - Sigla;Municipio;Rev

In [12]:
RENOMEAR = {
    "regiao_sigla": "regiao", "estado_sigla": "uf", "municipio": "municipio",
    "revenda": "revenda", "cnpj_da_revenda": "cnpj", "produto": "produto",
    "data_da_coleta": "data_coleta", "valor_de_venda": "valor_venda",
    "valor_de_compra": "valor_compra", "unidade_de_medida": "unidade",
    "bandeira": "bandeira",
}
ENDERECO = ["nome_rua", "numero_rua", "complemento", "bairro", "cep"]

def ler(caminho):
    nome, data = abrir_bytes(caminho)
    enc, sep, _ = detectar(data)
    linhas_brutas = data.count(b"\n")          # inclui o cabeçalho
    df = pd.read_csv(io.BytesIO(data), sep=sep, encoding=enc,
                     dtype=str, on_bad_lines="warn")
    df.columns = [limpa_nome(c) for c in df.columns]
    df = df.drop(columns=ENDERECO, errors="ignore").rename(columns=RENOMEAR)
    df["arquivo_origem"] = caminho.name
    print(f"{caminho.name}: {len(df):>9,} linhas lidas | "
          f"{linhas_brutas-1:>9,} no arquivo | colunas: {len(df.columns)}")
    return df

partes = [ler(a) for a in arquivos]

ca-2023-01.zip:   431,576 linhas lidas |   431,576 no arquivo | colunas: 13
ca-2023-02.zip:   472,424 linhas lidas |   472,424 no arquivo | colunas: 13
ca-2024-01.zip:   477,154 linhas lidas |   477,154 no arquivo | colunas: 13
ca-2024-02.zip:   421,382 linhas lidas |   421,382 no arquivo | colunas: 13
ca-2025-01.zip:   429,523 linhas lidas |   429,523 no arquivo | colunas: 13
ca-2025-02.zip:   384,208 linhas lidas |   384,208 no arquivo | colunas: 13
ca-2026-01.zip:   422,418 linhas lidas |   422,418 no arquivo | colunas: 13


In [13]:
esperadas = set(RENOMEAR.values())
for p in partes:
    print(p["arquivo_origem"].iloc[0], "faltando:", sorted(esperadas - set(p.columns)),
          "| extras:", sorted(set(p.columns) - esperadas - {"arquivo_origem"}))

df = pd.concat(partes, ignore_index=True)
del partes
print(df.shape)
df.head()

ca-2023-01.zip faltando: [] | extras: ['nome_da_rua']
ca-2023-02.zip faltando: [] | extras: ['nome_da_rua']
ca-2024-01.zip faltando: [] | extras: ['nome_da_rua']
ca-2024-02.zip faltando: [] | extras: ['nome_da_rua']
ca-2025-01.zip faltando: [] | extras: ['nome_da_rua']
ca-2025-02.zip faltando: [] | extras: ['nome_da_rua']
ca-2026-01.zip faltando: [] | extras: ['nome_da_rua']
(3038685, 13)


,regiao,uf,municipio,revenda,cnpj,nome_da_rua,produto,data_coleta,valor_venda,valor_compra,unidade,bandeira,arquivo_origem
0,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,ETANOL,03/01/2023,"3,39",NaN,R$ / litro,BRANCA,ca-2023-01.zip
1,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,DIESEL S10,03/01/2023,"6,21",NaN,R$ / litro,BRANCA,ca-2023-01.zip
2,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,GASOLINA,03/01/2023,"4,39",NaN,R$ / litro,BRANCA,ca-2023-01.zip
3,NE,AL,ARAPIRACA,COMERCIAL DE COMBUSTIVEIS E LUBRIFICANTES VITA...,02.817.655/0001-82,RODOVIA AL 220 - KM 06,ETANOL,02/01/2023,"3,89",NaN,R$ / litro,VIBRA ENERGIA,ca-2023-01.zip
4,NE,AL,ARAPIRACA,COMERCIAL DE COMBUSTIVEIS E LUBRIFICANTES VITA...,02.817.655/0001-82,RODOVIA AL 220 - KM 06,DIESEL,02/01/2023,"6,49",NaN,R$ / litro,VIBRA ENERGIA,ca-2023-01.zip


In [14]:
print(df.isna().sum())
print(df["produto"].value_counts())
print(df["unidade"].value_counts())
print(df["bandeira"].value_counts().head(15))

regiao               9114
uf                   9114
municipio            9114
revenda              9114
cnpj                 9114
nome_da_rua          9114
produto              9114
data_coleta          9114
valor_venda          9114
valor_compra      3038685
unidade              9114
bandeira             9114
arquivo_origem          0
dtype: int64
produto
GASOLINA              788453
ETANOL                666801
GASOLINA ADITIVADA    613318
DIESEL S10            556388
DIESEL                340500
GNV                    64111
Name: count, dtype: int64
unidade
R$ / litro    2965460
R$ / m³         55379
R$ / m3          8732
Name: count, dtype: int64
bandeira
BRANCA           1031171
IPIRANGA          641406
VIBRA             541156
RAIZEN            447124
VIBRA ENERGIA      99128
ALE                50257
ALESAT             47102
SABBÁ              31482
ATEM' S            13467
RODOIL             12957
CHARRUA            10930
RAIZEN MIME         9337
SP                  8247
EQUADOR

In [15]:
import os
INTER = pathlib.Path("/content/drive/MyDrive/precos-combustiveis-anp/intermediarios")
INTER.mkdir(exist_ok=True)
df.to_parquet(INTER / "unido_bruto.parquet", index=False)
print("salvo")

salvo
